# Partner Product Analysis

Exploratory look at the silver event layer: event mix, top products,
and a sanity check of the recommender's metrics. Expects the
streaming job to have written Parquet under `data/silver`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, desc

spark = SparkSession.builder.appName("partner-analysis").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

events = spark.read.parquet("../data/silver/partner_events")
print(f"rows: {events.count()}")
events.printSchema()

In [ ]:
# Event mix: views should dominate, purchases should be rare.
events.groupBy("event_type").agg(count("*").alias("n")).orderBy(desc("n")).show()

# Top products by purchase count.
events.filter(col("event_type") == "purchase").groupBy("product_id").agg(
    count("*").alias("purchases")).orderBy(desc("purchases")).show(10)

In [ ]:
import json
from pathlib import Path

# Metrics written by src/recommender.py
metrics_file = Path("../models/metrics.json")
if metrics_file.exists():
    print(json.dumps(json.loads(metrics_file.read_text()), indent=2))
else:
    print("No metrics yet -- run: python src/recommender.py")

In [ ]:
# Revenue by partner: which partners drive the most GMV?
events.filter(col("event_type") == "purchase").groupBy("partner_id").agg(
    {"revenue": "sum"}).withColumnRenamed("sum(revenue)", "gmv").orderBy(desc("gmv")).show()

spark.stop()